# 05 — Recorte de mano con MediaPipe + CNN

Este experimento estudia el efecto de localizar y recortar la mano antes de clasificarla. MediaPipe prepara los recortes; las arquitecturas CNN se mantienen separadas y configurables en `configs/`. La primera comparación recomendada es `baseline` frente a `baseline_handcrop`. Después se pueden ejecutar MobileNetV2 y EfficientNetB0 usando exactamente los mismos recortes.

> La evaluación end-to-end asigna `nothing` cuando MediaPipe no detecta una mano. Un fallo de detección en una seña cuenta como error. La métrica `test accuracy` estándar es condicional a que exista recorte; revisa `pipeline_metrics.json` para la métrica del flujo completo.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from src.train import train
from src.compare import latest_run, summary_table, pipeline_summary_table
print('Proyecto:', ROOT)

## 1. Preparar una sola vez los recortes

El script conserva la estructura de clases y nombres de archivo. Guarda un manifiesto con el estado de detección, necesario para medir cobertura y errores del flujo completo. Primero genera cinco ejemplos aleatorios por clase y revisa que el recorte incluya toda la mano. Si el resultado es razonable, ejecuta la celda de procesamiento completo. Los resultados quedan en `data/processed/`, excluido de Git.

In [ ]:
# Piloto rápido: inspecciona los recortes y la cobertura por clase antes del lote completo.
!python scripts/prepare_hand_crops.py --input data/raw/asl_alphabet_train/asl_alphabet_train --output data/processed/hand_crops/pilot --manifest data/processed/hand_crops/pilot_manifest.csv --max-per-class 5

Revisa visualmente `data/processed/hand_crops/pilot/` y las tasas de detección que imprimió el script. Si MediaPipe corta dedos o muñeca, ajusta `--margin` (por defecto 0.30) antes de procesar el dataset completo.

In [ ]:
# Ejecutar una vez después de revisar el piloto; omitir si ya existen los recortes completos.
!python scripts/prepare_hand_crops.py --input data/raw/asl_alphabet_train/asl_alphabet_train --output data/processed/hand_crops/asl_alphabet_train --manifest data/processed/hand_crops/asl_alphabet_train_manifest.csv

## 2. Entrenar las arquitecturas por separado

Cada modelo tiene su YAML y su corrida. Ejecuta de a uno, especialmente en el clúster, para controlar memoria y tiempo. Para medir específicamente el efecto del recorte, empieza con el baseline; luego compara los modelos preentrenados sobre los mismos recortes.

In [ ]:
run_baseline_crop = train('configs/baseline_handcrop.yaml', name='baseline_handcrop')

In [ ]:
# Ejecutar después si hay tiempo/recursos.
run_mobilenet_crop = train('configs/mobilenetv2_handcrop.yaml', name='mobilenetv2_handcrop')

In [ ]:
# Ejecutar después si hay tiempo/recursos.
run_efficientnet_crop = train('configs/efficientnetb0_handcrop.yaml', name='efficientnetb0_handcrop')

## 3. Comparar resultados

`metrics.json` resume el clasificador sobre ejemplos con mano detectada (más `nothing`). `pipeline_metrics.json` y `pipeline_confusion_matrix.png` incluyen fallos de detección como parte del sistema completo. No compares solo accuracy; revisa macro F1, matriz de confusión, detección en señas y tiempo por época.

In [ ]:
runs = {'CNN original': latest_run('baseline'), 'CNN + recorte': latest_run('baseline_handcrop')}
# Añade estas filas cuando terminen esas corridas:
# runs['MobileNetV2 + recorte'] = latest_run('mobilenetv2_handcrop')
# runs['EfficientNetB0 + recorte'] = latest_run('efficientnetb0_handcrop')
display(summary_table(runs))
pipeline_summary_table(runs)

## 4. Fotos propias

Prepara las fotos externas una sola vez y evalúalas con el script. Cambia `--run` por la corrida elegida. El CSV por imagen queda en `experiments/<corrida>/external_photos/`.

In [ ]:
!python scripts/prepare_hand_crops.py --input data/mis_fotos --output data/processed/hand_crops/mis_fotos --manifest data/processed/hand_crops/mis_fotos_manifest.csv

In [ ]:
!python scripts/evaluate_hand_crop.py --run experiments/AAAAMMDD-HHMM_baseline_handcrop